# EMASEO — Entrenamiento YOLOv8 en Google Colab T4
**Objetivo:** mAP@50 >= 0.85 | nc=1 | clase: garbage

## Antes de empezar
1. Menú superior: **Runtime > Change runtime type > T4 GPU**
2. Sube `dataset_entrenamiento.zip` a tu Google Drive (carpeta raíz o `Mi unidad/emaseo/`)
3. Ejecuta las celdas en orden

## Celda 1 — Verificar GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Celda 2 — Instalar dependencias

In [ ]:
!pip install ultralytics==8.3.* -q
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics {ultralytics.__version__} listo')

## Celda 3 — Montar Google Drive y extraer dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# Ajusta esta ruta si subiste el zip a una subcarpeta
ZIP_PATH = '/content/drive/MyDrive/dataset_entrenamiento.zip'
DATASET_DIR = '/content/dataset_entrenamiento'

if not os.path.exists(DATASET_DIR):
    print('Extrayendo dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Listo')
else:
    print('Dataset ya extraido')

# Contar imagenes
for split in ['train', 'valid', 'test']:
    imgs = list(Path(f'{DATASET_DIR}/{split}/images').glob('*')) if Path(f'{DATASET_DIR}/{split}/images').exists() else []
    print(f'  {split}: {len(imgs):,} imagenes')

## Celda 4 — Verificar data.yaml

In [ ]:
from pathlib import Path

DATASET_DIR = '/content/dataset_entrenamiento'
YAML_PATH = f'{DATASET_DIR}/data.yaml'

# Leer y mostrar
print(Path(YAML_PATH).read_text())

# Reescribir con path absoluto (necesario en Colab)
yaml_content = f"""path: {DATASET_DIR}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names:
  - garbage
"""
Path(YAML_PATH).write_text(yaml_content)
print('data.yaml actualizado con path absoluto')

## Celda 5 — ENTRENAMIENTO

> Esta celda tarda ~4-6 horas. No cierres el navegador.
> Tip: activa `Stay awake` en Chrome (extensión o `chrome://flags`) para evitar que Colab se desconecte.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # medium: mejor balance precision/velocidad para T4

results = model.train(
    data='/content/dataset_entrenamiento/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,          # T4 16GB aguanta batch=16 con yolov8m
    device=0,
    workers=4,
    patience=30,       # early stopping si no mejora en 30 epocas
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    # Augmentacion
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    # Paths
    project='/content/drive/MyDrive/emaseo_runs',  # guarda directo en Drive
    name='yolov8m_emaseo_v1',
    exist_ok=True,
    save=True,
    save_period=10,    # checkpoint cada 10 epocas
    amp=True,          # FP16 - funciona perfecto en T4
    verbose=True,
)

print(f'Mejor mAP@50:   {results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'Mejor mAP@50-95:{results.results_dict["metrics/mAP50-95(B)"]:.4f}')

## Celda 6 — Validacion final con best.pt

In [ ]:
from ultralytics import YOLO

best_path = '/content/drive/MyDrive/emaseo_runs/yolov8m_emaseo_v1/weights/best.pt'
model = YOLO(best_path)

metrics = model.val(
    data='/content/dataset_entrenamiento/data.yaml',
    imgsz=640,
    batch=16,
    device=0,
)

map50 = metrics.results_dict['metrics/mAP50(B)']
print(f'mAP@50:    {map50:.4f}')
print(f'Precision: {metrics.results_dict["metrics/precision(B)"]:.4f}')
print(f'Recall:    {metrics.results_dict["metrics/recall(B)"]:.4f}')

if map50 >= 0.85:
    print('OBJETIVO ALCANZADO: mAP@50 >= 0.85')
else:
    print(f'Falta {0.85 - map50:.4f} para llegar a 0.85 — considera mas epocas o fine-tuning')

## Celda 7 — Descargar best.pt a tu PC

In [ ]:
# Opcion A: ya esta en Drive (ruta arriba), solo copia la ruta y descarga desde Drive
# Opcion B: descargar directamente desde Colab
from google.colab import files
files.download('/content/drive/MyDrive/emaseo_runs/yolov8m_emaseo_v1/weights/best.pt')

## Notas
- El modelo se guarda en **Google Drive** cada 10 épocas — si Colab se desconecta no pierdes el progreso
- Para reanudar desde un checkpoint: `YOLO('ruta/epoch_X.pt').train(resume=True)`
- Si mAP@50 < 0.85 al final: aumenta `epochs` a 200 o reduce `patience` a 50